# L01 · Cargar un CSV y un TSV

| | |
|---|---|
| **Qué vas a aprender** | Leer ficheros de texto con los parámetros correctos |
| **Cuánto tiempo** | 45 min |
| **Datasets** | `ventas_2024_este.csv`, `ventas_2024_oeste.tsv` |

> **Al terminar vas a poder** abrir cualquier CSV que te pasen sin que se desmonte la
> tabla entera, y explicar por qué un fichero de 12.000 filas te puede salir con una
> sola columna.

In [ ]:
import sys
from pathlib import Path

# JupyterLab arrancado en la raíz del proyecto deja el cwd ahí.
# Si abres el notebook con el cwd en notebooks/, sube un nivel.
ROOT = Path.cwd()
if not (ROOT / "datos").exists() and (ROOT.parent / "datos").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks"))
DATOS = ROOT / "datos"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from herramientas import (cargar, mostrar, ficha, guardar_csv, guardar_excel,
                           guardar_figura, parsear_importes, parsear_fecha,
                           clave_canonica, detectar_alfabeto, auditar_fusiones,
                           resumen_entrega, perfil_memoria, contar_falsos_nulos)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
plt.style.use("seaborn-v0_8-whitegrid")
pd.__version__

## El concepto

`pd.read_csv` tiene una firma enorme, pero solo hay **cinco** parámetros que usas el
95% de las veces:

- `sep`: el delimitador. Si lo omites, pandas asume `,`.
- `encoding`: qué byte significa qué letra. Si lo omites, asume UTF-8.
- `decimal`: qué símbolo separa los decimales.
- `header`: en qué fila está la cabecera.
- `usecols`: qué columnas te interesan de verdad.

La lección de hoy no es la sintaxis. Es que **los tres primeros no se inventan: se
deducen del fichero**, y equivocarse no da error: da una tabla silenciosamente mal
leída.

## Manos a la obra

El primer paso SIEMPRE es mirar el fichero crudo. Nunca adivines:

In [ ]:
ruta = DATOS / "ventas_2024_este.csv"

with open(ruta, "rb") as f:
    primeros = f.read(220)

print(repr(primeros))

Fíjate en dos cosas: los `;` como separador, y los bytes `\x..` (por ejemplo
`\xf3`, que es la `ó` en latin-1). Esos bytes no son UTF-8 válido. El `repr`
enseña los bytes, no el texto ya decodificado.

Es un export de un ERP español: `;` como delimitador, coma decimal, fechas `dd/mm/yyyy`
y cabeceras con espacios y tildes.

In [ ]:
este = pd.read_csv(
    DATOS / "ventas_2024_este.csv",
    sep=";",            # <-- separador de punto y coma
    decimal=",",         # <-- coma decimal (formato europeo)
    encoding="latin-1",  # <-- no es UTF-8
)

print("shape:", este.shape)
este.dtypes.head(6)

In [ ]:
# El MISMO ERP, otra región, otro formato: tabulador, punto decimal, fechas ISO.
oeste = pd.read_csv(DATOS / "ventas_2024_oeste.tsv", sep="\t", encoding="utf-8")

print("shape:", oeste.shape)
mostrar(oeste, 5, "TSV de la rama oeste")

## ⚠️ La trampa

### Trampa 1: olvidar el `sep`, y no salta ningún error

Esto **no** da error. Da una tabla con una sola columna:

In [ ]:
roto = pd.read_csv(DATOS / "ventas_2024_este.csv", encoding="latin-1")
print("shape:", roto.shape)   # (12000, 1) en vez de (12000, 11)
roto.columns

No hay excepción, no hay warning. Solo una columna llamada
`ID Pedido;Fecha Pedido;ID Cliente;...` que parece tu data pero no lo es. Todos los
`groupby`, `merge` y `describe` posteriores dan resultados sin sentido.

### Trampa 2: `decimal=','` sin `sep=';'` (o al revés)

Cambiar solo uno de los dos rompe, y de forma más sutil:

In [ ]:
print("con decimal=  :", este["Precio Unitario"].head(3).tolist())

sin_decimal = pd.read_csv(
    DATOS / "ventas_2024_este.csv", sep=";", encoding="latin-1"
)
print("sin decimal=  :", sin_decimal["Precio Unitario"].head(3).tolist())

En este fichero el precio está guardado como `773,02`, sin punto de miles. Sin
`decimal=","` pandas **no** lo convierte en `77302`: lo deja como texto
`'773,02'`. El fallo es otro, y también es silencioso: la columna no es numérica,
así que un `.sum()` no es una facturación.

El salto de magnitud sí existe, con otro patrón. `1.234,56` son 1234.56 €. Si
alguien borra la coma pensando que es un separador de miles, se queda en `1.234`.

In [ ]:
print("dtype con decimal   :", este["Precio Unitario"].dtype)
print("dtype sin decimal   :", sin_decimal["Precio Unitario"].dtype)

ejemplo = "1.234,56"
print("bien (coma decimal) :", float(ejemplo.replace(".", "").replace(",", ".")))
print("mal  (coma borrada) :", float(ejemplo.replace(",", "")))

In [ ]:
try:
    pd.read_csv(DATOS / "ventas_2024_este.csv", sep=";", decimal=",")
except UnicodeDecodeError as e:
    print("UnicodeDecodeError:", str(e)[:70], "...")
    print("-> sin encoding, falla. con OTRO encoding, no falla: falla peor,")
    print("   porque le pasa todo a latin y destroza los acentos en silencio.")

## 🧪 Tu turno

- **Ejercicio 1** — Carga el fichero de finanzas y responde con `print()`: cuántas filas
  y columnas tiene, cuál es su separador y su encoding (míralos primero con `read()`).
- **Ejercicio 2** — Lee `productos_catalogo.csv` usando solo
  `usecols=["sku", "descripcion", "precio"]` y comprueba con `memory_usage` que ocupa
  menos que leerlo entero.
- **Ejercicio 3** — ¿Por qué `este.shape` y `oeste.shape` tienen columnas distintas?
  Nombra las 4 diferencias de esquema que ves entre ambos ficheros.

## 📦 Workbook integrador: auditoría de carga

**Contexto.** Eres el responsable de datos de un retailer. Los dos exportadores del ERP
mandan ficheros a dos equipos distintos, y cada equipo abre su fichero con los
parámetros que cree convenientes. Tu jefa pide un informe de qué se ha cargado bien.

**Tu trabajo.** Escribir una función que, para cada fichero de `datos/`, detecte
automáticamente: nº de filas, nº de columnas, delimitador real, encoding real y **3
anomalías concretas**. Sin que le pases los parámetros a mano: tiene que deducirlos del
propio fichero.

**Restricciones.** Nada de `try/except` que se trague errores: si un fichero no se puede
leer, tiene que decir por qué.

In [ ]:
import re

def detectar_separador(ruta: Path) -> str:
    """Deduce el delimitador contando candidatos en la cabecera."""
    if ruta.suffix == ".xlsx":
        return "(excel)"
    cabecera = ruta.read_bytes()[:4000].decode("latin-1", errors="replace").splitlines()[0]
    candidatos = [s for s in [";", ",", "\t", "|"] if s in cabecera]
    if not candidatos:
        raise ValueError(f"No encuentro separador en {ruta.name}")
    return max(candidatos, key=cabecera.count)


def detectar_encoding(ruta: Path) -> str:
    """BOM primero. latin-1 decodifica cualquier byte, así que no puede ir el primero."""
    if ruta.suffix == ".xlsx":
        return "(xlsx)"
    datos = ruta.read_bytes()[:8000]
    if datos.startswith(b"\xef\xbb\xbf"):
        return "utf-8-sig"
    try:
        datos.decode("utf-8")
    except UnicodeDecodeError:
        # 0x80-0x9F son letras en cp1252 y controles en latin-1.
        if any(0x80 <= b <= 0x9F for b in datos):
            return "cp1252"
        return "latin-1"
    return "utf-8"


def detectar_decimal(ruta: Path, sep: str, enc: str) -> str:
    """Mira el texto. Un ';' no implica coma decimal: finanzas mezcla los dos."""
    if ruta.suffix == ".xlsx":
        return ""
    muestra = pd.read_csv(ruta, sep=sep, encoding=enc, dtype=str, nrows=400)
    texto = " ".join(muestra.fillna("").astype(str).to_numpy().ravel())
    coma = len(re.findall(r"\d,\d", texto))
    punto = len(re.findall(r"\d\.\d", texto))
    if coma and punto:
        return "mixto"
    if coma:
        return ","
    if punto:
        return "."
    return ""


def auditoria(ruta: Path) -> dict:
    """Carga autodetectando y devuelve una fila de diagnóstico."""
    sep = detectar_separador(ruta)
    enc = detectar_encoding(ruta)
    if ruta.suffix == ".xlsx":
        df = pd.read_excel(ruta, index_col=0)
        decimal = ""
    else:
        decimal = detectar_decimal(ruta, sep, enc)
        kwargs = {"sep": sep, "encoding": enc}
        if decimal in {",", "."}:
            kwargs["decimal"] = decimal
        df = pd.read_csv(ruta, **kwargs)

    anomalias = []
    if decimal == "mixto":
        anomalias.append("decimales mixtos en el mismo fichero")
    for col, n in df.isna().sum().items():
        if n:
            anomalias.append(f"{col}: {n} nulos")
    if df.duplicated().any():
        anomalias.append(f"{int(df.duplicated().sum())} filas duplicadas")
    # include="str" es la forma nueva (pandas 3); include="object" es la vieja.
    try:
        texto_cols = df.select_dtypes(include="str").columns
    except TypeError:
        texto_cols = df.select_dtypes(include="object").columns
    for col in texto_cols:
        if df[col].astype(str).str.contains(r"\s{2,}").any():
            anomalias.append(f"{col}: espacios dobles")

    return {
        "fichero": ruta.name,
        "filas": len(df),
        "columnas": df.shape[1],
        "separador": repr(sep),
        "encoding": enc,
        "decimal": decimal or "(no aplica)",
        "memoria_kb": round(df.memory_usage(deep=True).sum() / 1024, 1),
        "anomalias": " | ".join(anomalias[:3]) or "ninguna",
    }

In [ ]:
ficheros = (
    sorted(DATOS.glob("*.csv"))
    + sorted(DATOS.glob("*.tsv"))
    + sorted(DATOS.glob("*.xlsx"))
)
informe = pd.DataFrame([auditoria(f) for f in ficheros])

mostrar(informe, n=10, titulo="AUDITORÍA DE CARGA")

In [ ]:
ruta_informe = guardar_csv(informe, "L01_auditoria_carga")
resumen_entrega(ruta_informe)

### Criterios de aceptación

1. `auditoria()` funciona sobre los **8** ficheros sin intervención manual.
2. Detecta que `ventas_2024_este.csv` usa `';'` + `','` + `latin-1`, que
   `ventas_2024_oeste.tsv` usa tabulador + `'.'` + `utf-8`, y que
   `clientes_crm.csv` es `utf-8-sig` (tiene BOM). `finanzas_conciliacion.csv`
   sale como decimal `mixto`: un `';'` no autoriza a forzar la coma.
3. `informe.csv` tiene una fila por fichero, y la columna `anomalias` no está vacía en
   al menos 6 de ellos.
4. Se escribe `salida/L01_auditoria_carga.csv`.